# Run TWave on a recording

Point this at an EEG file (`.npy` or `.npz`), set the sample rate, and run the cells top to bottom.

**You'll get:**
1. A detection log (CSV) — every stim with its sample index and time
2. A 3-panel summary figure — stim-triggered average, stim-phase polar plot, status counts
3. One plot per detected stim with ±4s of context around it

**Known limitation:** TWave's default parameters are tuned for the intracranial data Johannes was working with. If you see mostly `INHIBITED_AMP` in the status counts, the amplitude gates are wrong for your data — scroll to the CONFIG cell and adjust `amp_threshold_uv` and `amp_limit_uv`. The diagnostic output at the top of the run cell will tell you which way to move them.

## Setup — run once per session

In [ ]:
import sys, os
from pathlib import Path

# Where is the upstream t-wave-algo folder? Resolve relative to repo root.
# This notebook is expected to live at johannes-code/validation/.
REPO_ROOT = Path(os.getcwd()).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / 't-wave-algo'))

import numpy as np
import matplotlib.pyplot as plt
import pickle, csv
from math import pi
from collections import Counter
from scipy.signal import butter, sosfilt

from Simulations import SimulationDataset, SimulationResult, PhaseTrackerStatus
from Algo_TWave import PhaseTracker as TWave

print(f'Repo root: {REPO_ROOT}')

## CONFIG — set these for your file

In [ ]:
# ── File ───────────────────────────────────────────────────────────────
# Can be .npy (raw array) or .npz (with 'data', 'fs', 'scale_factors' — see
# ns6_to_npz.py). For .npy, you MUST set `fs` below.
input_path = REPO_ROOT / 'data' / 'annotated' / 'Patient03_Channel1_EEG.npy'

# ── Sample rate (Hz) ───────────────────────────────────────────────────
# Ignored for .npz (fs is read from the file).
# TWave's defaults are tuned for ~500 Hz — stay near that range.
fs = 512

# ── Channel (only for multi-channel files) ─────────────────────────────
channel_idx = 0

# ── Duration ───────────────────────────────────────────────────────────
max_duration_s = 0        # 0 = use whole file; otherwise process first N seconds

# ── TWave amplitude gates (µV) ─────────────────────────────────────────
# These get overridden on the TWave instance. If your signal has very different
# amplitudes (e.g. scalp EEG vs intracranial), adjust — the run cell prints
# your signal's percentiles so you can set these sensibly.
amp_threshold_uv = 75     # lower bound — below this, INHIBITED_AMP fires
amp_limit_uv     = 3000   # upper bound — above this, INHIBITED_AMP fires

# ── Output paths ───────────────────────────────────────────────────────
output_dir = REPO_ROOT / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

# ── Event-inspection plot knobs ────────────────────────────────────────
context_s = 4.0           # ± seconds of context around each detection
max_events_to_plot = 50   # cap on per-event plots (use `None` for all)

# ── Progress reporting ─────────────────────────────────────────────────
progress_every_pct = 5

## Load file

In [ ]:
def load_signal(path, fs_hint, channel_idx=0, max_duration_s=0):
    """Load a signal from .npy or .npz. Returns (sig_uv, fs).

    - .npy: array is assumed to already be in µV; fs_hint is used.
    - .npz: expected to have keys 'data', 'fs', optionally 'scale_factors'.
            Matches the format produced by ns6_to_npz.py.
    """
    path = Path(path)
    if path.suffix == '.npy':
        arr = np.load(path)
        print(f'  .npy loaded: shape={arr.shape}, dtype={arr.dtype}')
        if arr.ndim == 1:
            sig = arr
        elif arr.ndim == 2:
            # assume longer axis is time
            sig = arr[:, channel_idx] if arr.shape[0] >= arr.shape[1] else arr[channel_idx, :]
        else:
            raise ValueError(f'Unsupported array shape {arr.shape}')
        return np.asarray(sig, dtype=np.float64).squeeze(), fs_hint

    elif path.suffix == '.npz':
        npz = np.load(path, allow_pickle=False)
        print(f'  .npz keys: {list(npz.files)}')
        data = npz['data']
        fs_file = float(npz['fs'])
        print(f'  fs from file: {fs_file} Hz (overriding fs_hint={fs_hint})')
        sig = data[:, channel_idx] if data.ndim == 2 else data
        if np.issubdtype(sig.dtype, np.integer) and 'scale_factors' in npz.files:
            sf = float(npz['scale_factors'][channel_idx])
            sig = sig.astype(np.float64) * sf
            print(f'  scaled int16 → µV using factor {sf:.6g}')
        else:
            sig = sig.astype(np.float64)
        return sig.squeeze(), fs_file

    else:
        raise ValueError(f'Unsupported file extension: {path.suffix}')


sig, fs_loaded = load_signal(input_path, fs_hint=fs, channel_idx=channel_idx)
fs = fs_loaded  # use whatever the file / config says

if max_duration_s > 0:
    sig = sig[: int(max_duration_s * fs)]

print(f'\nSignal: {len(sig)} samples, {len(sig)/fs:.1f} s at {fs} Hz')

## Run TWave

In [ ]:
def run_simulations_sane(dataset, phase_tracker, block_size_sp=12, progress_every_pct=5):
    """Drop-in replacement for Simulations.run_simulations.
    The upstream version prints progress on every block (buggy condition).
    This one reports every `progress_every_pct` percent.
    """
    signal = np.nan_to_num(dataset.signal, nan=0.0)
    n = len(signal)
    stims_sp, internals_ts, status_ts = [], [], []
    next_report = progress_every_pct

    for i in range(0, n, block_size_sp):
        block = signal[i:i + block_size_sp]
        for si, sample in enumerate(block):
            result, internals = phase_tracker.update(sample)
            if result.status & PhaseTrackerStatus.STIM1:
                stim_idx = i + si + result.delay_sp
                stims_sp.append(stim_idx)
                print(f'  STIM1 @ sample {stim_idx}  ({stim_idx/dataset.fs:.2f}s)')
            status_ts.append(result.status)
            internals_ts.append(internals)
        pct = 100 * i / n
        if pct >= next_report:
            print(f'progress: {pct:.0f}%')
            next_report += progress_every_pct
    print('progress: 100%')
    stims_sp = [x for x in stims_sp if 0 <= x < n]
    return SimulationResult(dataset, phase_tracker, stims_sp, status_ts, internals_ts)


# Diagnostic — make sure amp gates are in the right ballpark
p50, p90, p99 = np.percentile(np.abs(sig), [50, 90, 99])
print(f'|signal| percentiles: p50={p50:.1f}  p90={p90:.1f}  p99={p99:.1f} µV')
print(f'TWave gates:          amp_threshold={amp_threshold_uv}  amp_limit={amp_limit_uv} µV')
if amp_threshold_uv > p90:
    print('  ⚠  amp_threshold_uv is above the 90th percentile — most samples will be INHIBITED_AMP')
if amp_limit_uv < p99:
    print('  ⚠  amp_limit_uv is below the 99th percentile — many samples will be INHIBITED_AMP')

ds_sim = SimulationDataset(
    t=np.arange(len(sig)) / fs,
    signal=sig,
    fs=fs,
    name=Path(input_path).stem + f'_ch{channel_idx}_fs{fs}',
)

# Upstream TWave's kwargs override has a print-formatting bug; set attrs directly.
tw = TWave(ds_sim.fs)
tw.amp_threshold_uv = amp_threshold_uv
tw.amp_limit_uv = amp_limit_uv
print(f'TWave overrides: amp_threshold_uv={amp_threshold_uv}  amp_limit_uv={amp_limit_uv}\n')

rslt = run_simulations_sane(ds_sim, tw, progress_every_pct=progress_every_pct)

print(f'\nFound {len(rslt.stims_sp)} stims.')

## Status breakdown

If you got few/no stims, the dominant status tells you why. `INHIBITED_AMP` = amplitude gates wrong. `INHIBITED_QUADRATURE` = signal doesn't look sinusoidal enough. `INHIBITED_RATIO` = too much high-frequency content relative to low. Each has a corresponding threshold in `Algo_TWave.py` you can override the same way as the amp gates.

In [ ]:
counts = Counter(rslt.status_ts)
total = sum(counts.values())
print(f'{"status":<55s} {"count":>10s} {"%":>7s}')
print('-' * 74)
for s, n in sorted(counts.items(), key=lambda x: -x[1]):
    name = PhaseTrackerStatus(int(s)).name if s != 0 else 'NONE'
    print(f'{name:<55s} {n:>10d} {100*n/total:>6.1f}%')

## Save detection log

In [ ]:
stem = Path(input_path).stem
csv_path = output_dir / f'{stem}_stims.csv'

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['stim_index', 'sample_idx', 'time_s', 'est_freq_hz', 'est_amp_uv', 'est_phase_rad'])
    for i, s_idx in enumerate(rslt.stims_sp):
        internals = rslt.internals_ts[s_idx] if s_idx < len(rslt.internals_ts) else {}
        writer.writerow([
            i + 1,
            s_idx,
            f'{s_idx/fs:.4f}',
            f'{internals.get("freq", float("nan")):.3f}',
            f'{internals.get("amp", float("nan")):.1f}',
            f'{internals.get("phase", float("nan")):.3f}',
        ])

print(f'Saved {len(rslt.stims_sp)} stims → {csv_path}')

# Also pickle the full result for downstream notebooks
pkl_path = output_dir / f'{stem}_result.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(rslt, f)
print(f'Pickled result → {pkl_path}')

## Summary figure

Three panels: stim-triggered average (SO-band), polar histogram of stim phases, status-count bar. Read left-to-right for a sense of whether detections are firing at real slow waves and at the right phase.

In [ ]:
SO_FREQ_RANGE = (0.5, 2.0)
WIN_S = 2.0
TARGET_PHASE = 0.0

# filter the signal to the SO band for plotting
nyq = fs / 2.0
sos_so = butter(4, [SO_FREQ_RANGE[0]/nyq, SO_FREQ_RANGE[1]/nyq], btype='band', output='sos')
sig_so = sosfilt(sos_so, sig)

win_samples = int(WIN_S * fs)
t_win = np.arange(-win_samples, win_samples) / fs

stim_windows = []
for s_idx in rslt.stims_sp:
    lo, hi = s_idx - win_samples, s_idx + win_samples
    if lo >= 0 and hi < len(sig_so):
        stim_windows.append(sig_so[lo:hi])

stim_phases = []
if stim_windows:
    centre_idx = len(stim_windows[0]) // 2
    half_cycle = int(fs / 2)
    lo_w = max(0, centre_idx - half_cycle)
    hi_w = min(len(stim_windows[0]), centre_idx + half_cycle)
    for w in stim_windows:
        peak_idx = lo_w + np.argmax(w[lo_w:hi_w])
        dt = (peak_idx - centre_idx) / fs
        stim_phases.append((-dt * 2 * pi) % (2 * pi))
    stim_phases = np.array(stim_phases)

fig = plt.figure(figsize=(18, 5))
fig.suptitle(ds_sim.name, fontsize=12, fontweight='bold')

# (a) stim-triggered average
ax_a = fig.add_subplot(1, 3, 1)
if stim_windows:
    for w in stim_windows:
        ax_a.plot(t_win, w, alpha=0.3, lw=0.5)
    ax_a.plot(t_win, np.mean(stim_windows, axis=0), 'k-', lw=2, label='mean')
ax_a.axvline(0, color='red', ls='--', alpha=0.6, lw=1, label='stim')
ax_a.set_xlabel('Time from stim (s)'); ax_a.set_ylabel('Amplitude (SO band, µV)')
ax_a.set_title(f'Stim-triggered average (n={len(stim_windows)})')
ax_a.legend(fontsize=8)

# (b) polar phase histogram
ax_b = fig.add_subplot(1, 3, 2, projection='polar')
if len(stim_phases) > 0:
    n_bins = 24
    counts_b, edges = np.histogram(stim_phases, bins=n_bins, range=(0, 2*pi))
    centres = (edges[:-1] + edges[1:]) / 2
    ax_b.bar(centres, counts_b, width=2*pi/n_bins, alpha=0.6, edgecolor='k',
             lw=0.5, color='#E74C3C')
    mean_phase = np.angle(np.mean(np.exp(1j*stim_phases))) % (2*pi)
    ax_b.axvline(TARGET_PHASE, color='green', ls='--', lw=2)
    ax_b.axvline(mean_phase, color='darkred', ls='-', lw=2)
    ax_b.set_thetagrids([0, 90, 180, 270],
                        labels=['peak', 'falling', 'trough', 'rising'], fontsize=9)
    ax_b.set_yticklabels([])
    err_deg = abs(mean_phase - TARGET_PHASE) * 180 / pi
    if err_deg > 180: err_deg = 360 - err_deg
    ax_b.set_title(f'Stim phase (n={len(stim_phases)})\n'
                   f'green=target  red=actual  error={err_deg:.0f}°', fontsize=9, pad=15)
else:
    ax_b.set_title('No stims')

# (c) status counts (top 6)
ax_c = fig.add_subplot(1, 3, 3)
top = sorted(counts.items(), key=lambda x: -x[1])[:6]
labels = [PhaseTrackerStatus(int(s)).name if s != 0 else 'NONE' for s, _ in top]
vals = [n for _, n in top]
labels_wrapped = ['\n'.join(l.split('|')) for l in labels]
ax_c.barh(range(len(top)), vals, color='steelblue', edgecolor='k', lw=0.5)
ax_c.set_yticks(range(len(top)))
ax_c.set_yticklabels(labels_wrapped, fontsize=7)
ax_c.invert_yaxis()
ax_c.set_xlabel('Sample count')
ax_c.set_title(f'Top 6 statuses (of {total} samples)')

plt.tight_layout()
fig_path = output_dir / f'{stem}_summary.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## Per-event plots

One figure per detected stim. Raw signal in grey, SO-band in blue, stim marked in red. The title shows sample index + TWave's internal freq/amp/phase estimates at that point.

In [ ]:
ctx_samples = int(context_s * fs)
t_full = np.arange(len(sig)) / fs

n_plot = len(rslt.stims_sp) if max_events_to_plot is None else min(len(rslt.stims_sp), max_events_to_plot)
print(f'Plotting {n_plot} of {len(rslt.stims_sp)} stims')

for i, s_idx in enumerate(rslt.stims_sp[:n_plot]):
    t_stim = s_idx / fs
    lo = max(0, s_idx - ctx_samples)
    hi = min(len(sig), s_idx + ctx_samples)

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(t_full[lo:hi], sig[lo:hi], color='lightgrey', lw=0.5,
            label='raw' if i == 0 else '')
    ax.plot(t_full[lo:hi], sig_so[lo:hi], color='steelblue', lw=1,
            label='SO band' if i == 0 else '')
    ax.axvline(t_stim, color='red', ls='--', lw=1.5, alpha=0.7,
               label='stim' if i == 0 else '')

    internals = rslt.internals_ts[s_idx] if s_idx < len(rslt.internals_ts) else {}
    freq = internals.get('freq', float('nan'))
    amp = internals.get('amp', float('nan'))
    phase = internals.get('phase', float('nan'))

    ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'#{i+1}  sample={s_idx}  t={t_stim:.2f}s  '
                 f'f={freq:.2f}Hz  amp={amp:.0f}µV  phase={phase:.2f}rad',
                 fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()